# Lab 4: Graphs, Cycles & Recovery — LangGraph Workflows

**Series**: Agentic AI Science Playbook — Foundation Layer
**Goal**: Model agent logic as a directed graph with branching, cycles, and error recovery.
**Time**: ~60 min.

## Why Graphs?

```
 [query_planner] --> [search] --(results)--> [summarize] --> [answer] --> END
                        |
                    (no results)
                        v
                   [refine_query] --> [search]   (cycle)
                        |
                  (max retries)
                        v
                     [error] --> END
```

## Learning Objectives

By the end of this lab, you will be able to:
- Model agent workflows as directed graphs using LangGraph
- Implement conditional routing based on intermediate results
- Build retry cycles for error recovery
- Design state machines that handle both success and failure paths

## Why This Matters for Scientists

Real scientific workflows are not linear. A literature search might return no results, requiring a broader query. A simulation might fail, requiring different parameters. An experiment might produce unexpected results, requiring a revised hypothesis.

**LangGraph** lets you model these branching, looping, recovering workflows as **state machines** — directed graphs where nodes are actions and edges are decisions.

> **Real-world example**: In drug discovery, a molecular generation → docking → scoring pipeline might fail at the docking stage. Instead of crashing, a graph-based agent can automatically retry with different parameters, try an alternative docking method, or escalate to human review. This is the pattern used in NVIDIA's BioNeMo-powered drug discovery pipelines.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Labs 0-3 |
| New libraries | `langgraph`, `langchain` |

**NVIDIA Connection**: **LangGraph** is the orchestration layer used by NVIDIA's **NeMo Agent Toolkit** for building production agent workflows. The state machine patterns you learn here are directly applicable when building agents with NIM endpoints and domain SDKs like BioNeMo, Earth-2, and PhysicsNeMo.

### Install Dependencies
We add `langgraph` and `langchain` for graph-based agent orchestration.

In [ ]:
!pip install -q openai langgraph langchain langchain-openai

### Connect to LLM
Same dual-provider setup used across all labs. Detects whether to use NVIDIA NIM or OpenAI based on available environment variables.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")

## Warm-Up: Your First Graph in 3 Minutes

Before we build the full research graph, let's understand LangGraph with the **simplest possible example** — a 2-node graph that takes a name and says hello.

If you've used Lab 3's Python classes, think of LangGraph as:
- **State** = a dictionary (like Lab 3's memory stores)
- **Node** = a function that reads and updates the state (like Lab 3's `run_persistent_agent`)
- **Edge** = the order functions run in (like calling functions in sequence)

That's it. Let's see it in action.

In [ ]:
# === WARM-UP: Simplest possible LangGraph ===
from typing import TypedDict
from langgraph.graph import StateGraph, END

# Step 1: Define state — just a dictionary with typed fields
class GreetingState(TypedDict):
    name: str
    greeting: str

# Step 2: Define nodes — functions that read/update state
def say_hello(state):
    return {"greeting": f"Hello, {state['name']}! Welcome to LangGraph."}

# Step 3: Build the graph — connect nodes with edges
graph = StateGraph(GreetingState)
graph.add_node("say_hello", say_hello)
graph.set_entry_point("say_hello")
graph.add_edge("say_hello", END)

# Step 4: Compile and run
app = graph.compile()
result = app.invoke({"name": "Scientist", "greeting": ""})
print(result["greeting"])

### What Just Happened?

That's the entire LangGraph pattern in 10 lines:

| Step | What you wrote | What it does |
|------|---------------|-------------|
| `TypedDict` | `class GreetingState` | Defines what data flows through the graph |
| `def say_hello(state)` | Node function | Reads state, returns updates |
| `graph.add_node()` | Register the node | Tells the graph this function exists |
| `graph.set_entry_point()` | First node | Where execution starts |
| `graph.add_edge("say_hello", END)` | Connection | After say_hello, stop |
| `graph.compile().invoke()` | Run it | Execute the graph with initial state |

> **Key insight**: A LangGraph node is just a Python function that takes a dict and returns a dict. If you can write `def f(state): return {"key": "value"}`, you can write LangGraph nodes.

Now let's add a **second node** and a **conditional edge** — the two concepts that make graphs powerful.

In [ ]:
# === WARM-UP Part 2: Two nodes + conditional edge ===

class ResearchState(TypedDict):
    question: str
    has_answer: bool
    answer: str

def check_knowledge(state):
    """Node 1: Do we already know the answer?"""
    known_topics = ["CRISPR", "PCR", "DNA"]
    has_answer = any(topic in state["question"].upper() for topic in known_topics)
    return {"has_answer": has_answer}

def answer_question(state):
    """Node 2a: We know this topic — answer directly."""
    return {"answer": f"I know about this! Here's what I know about '{state['question'][:30]}...'"}

def search_first(state):
    """Node 2b: We don't know — search first."""
    return {"answer": f"I need to search for '{state['question'][:30]}...' before answering."}

def route_by_knowledge(state):
    """Conditional edge: choose next node based on state."""
    return "answer" if state["has_answer"] else "search"

# Build the graph
g = StateGraph(ResearchState)
g.add_node("check", check_knowledge)
g.add_node("answer", answer_question)
g.add_node("search", search_first)

g.set_entry_point("check")
g.add_conditional_edges("check", route_by_knowledge, {"answer": "answer", "search": "search"})
g.add_edge("answer", END)
g.add_edge("search", END)

app2 = g.compile()

# Test with known topic
r1 = app2.invoke({"question": "How does CRISPR work?", "has_answer": False, "answer": ""})
print(f"Known topic:   {r1['answer']}")

# Test with unknown topic
r2 = app2.invoke({"question": "What is quantum gravity?", "has_answer": False, "answer": ""})
print(f"Unknown topic: {r2['answer']}")

### Warm-Up Complete!

You just learned the three building blocks of LangGraph:

| Building Block | What It Does | Example |
|---------------|-------------|---------|
| **Node** | A function that processes state | `check_knowledge`, `answer_question` |
| **Edge** | Fixed connection between nodes | `answer → END` |
| **Conditional edge** | Choose next node based on state | `if has_answer → answer, else → search` |

With these three pieces, you can build any workflow — including **retry loops** (a node that goes back to a previous node) and **error recovery** (a conditional edge that routes to an error handler).

That's exactly what we'll build next: a full research pipeline with search, summarize, retry, and error handling.

---

## Concept: Why Graphs Instead of Sequential Code?

Compare these two approaches to a research pipeline:

**Sequential** (fragile):
```python
results = search(query)         # What if this returns nothing?
summary = summarize(results)    # Crashes on empty input
answer = generate_answer(summary)
```

**Graph-based** (resilient):
```
search → [has results?]
           ├─ YES → summarize → answer → END
           └─ NO  → [retries < 3?]
                      ├─ YES → refine_query → search (cycle)
                      └─ NO  → error_message → END
```

The graph approach handles **every possible outcome** at every step. This is essential for scientific agents where:
- Databases may be unavailable
- Searches may return no results
- Simulations may fail or timeout
- Results may need human validation

### LangGraph Core Concepts

| Concept | Definition | Example |
|---------|-----------|---------|
| **State** | Data flowing through the graph | `{query, results, summary, error, retry_count}` |
| **Node** | A function that transforms state | `search_node`, `summarize_node` |
| **Edge** | Connection between nodes | `search → summarize` |
| **Conditional edge** | Edge chosen by a routing function | `if results: → summarize else: → refine` |
| **Cycle** | Back-edge creating a loop | `refine → search` (retry loop) |

### Define the State Schema
LangGraph uses a typed state dictionary to pass data between nodes. Every field represents a piece of information that flows through the graph — from the user's query to the final answer.

## State Definition

In [ ]:
from typing import TypedDict

class ResearchState(TypedDict):
    user_query: str
    refined_query: str
    search_results: list
    summary: str
    final_answer: str
    error: str
    retry_count: int

def fresh_state(q):
    return ResearchState(user_query=q, refined_query="", search_results=[],
                         summary="", final_answer="", error="", retry_count=0)

print("State fields:", list(ResearchState.__annotations__))

## Node Functions

### LLM Helper
A simple wrapper that sends a prompt to the LLM and returns the response text. All nodes use this.

In [ ]:
def llm_call(prompt, max_tokens=300):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}])
    return (r.choices[0].message.content or "").strip()

### Node: Query Planner
Refines the user's question into a precise search query (max 15 words). This is the first node in our graph.

In [ ]:
def query_planner_node(state):
    print(f"  [planner] '{state['user_query'][:50]}'")
    refined = llm_call(
        f"Convert to a precise scientific search query (max 15 words):\nQuestion: {state['user_query']}\nQuery:", 40)
    return {"refined_query": refined}

### Node: Search
Searches for papers matching the refined query. Returns results or an empty list. Notice: the 'obscure' keyword simulates a failed search to test our retry logic.

In [ ]:
def search_node(state):
    q = state["refined_query"] or state["user_query"]
    print(f"  [search] '{q[:50]}'")
    if state["retry_count"] == 0 and "obscure" in q.lower():
        print("  [search] No results.")
        return {"search_results": [], "retry_count": state["retry_count"]+1}
    return {
        "search_results": [f"Paper A on '{q[:20]}...' — Smith et al. (2023)",
                           f"Paper B on '{q[:20]}...' — Jones et al. (2024)"],
        "retry_count": state["retry_count"]+1,
    }

### Node: Refine Query
If search returned no results, this node asks the LLM to broaden the query. It's part of the retry cycle.

In [ ]:
def refine_query_node(state):
    print(f"  [refine] retry={state['retry_count']}")
    broader = llm_call(
        f"This query failed: '{state['refined_query']}'\nWrite a broader query (max 10 words):", 25)
    return {"refined_query": broader}

### Nodes: Summarize, Answer, and Error
Three output nodes: `summarize` extracts key takeaways, `answer` generates a final response, and `error` handles the case where all retries failed.

In [ ]:
def summarize_node(state):
    print(f"  [summarize] {len(state['search_results'])} results")
    text = "\n".join(state["search_results"])
    summary = llm_call(f"3 key takeaways from:\n{text}", 200)
    return {"summary": summary}

def answer_node(state):
    print("  [answer]")
    answer = llm_call(
        f"Q: {state['user_query']}\nSummary: {state['summary']}\nAnswer concisely:", 250)
    return {"final_answer": answer}

def error_node(state):
    return {"final_answer": f"No results for: '{state['user_query']}'. Please rephrase."}

## Concept: Conditional Routing — The Brain of the Graph

The `route_after_search` function is where intelligence meets architecture. It examines the current state and decides the next step:

```python
def route_after_search(state):
    if state["search_results"]:   return "summarize"  # Success path
    if state["retry_count"] >= 3: return "error"      # Give up path
    return "refine"                                     # Retry path
```

This is a **policy function** — it encodes your domain knowledge about how to handle different outcomes. In scientific agents, these routing decisions often involve:

- **Confidence thresholds**: "Is this result good enough, or should we try again?"
- **Resource limits**: "Have we spent too many API calls on this query?"
- **Safety checks**: "Should a human review this before we proceed?"

### Build the Graph
Now we wire up all nodes with edges. The key is `add_conditional_edges()` — after search, we check: did we get results? If yes, go to summarize. If no and retries left, refine and retry. If no and retries exhausted, go to error.

## Routing and Graph Construction

In [ ]:
from langgraph.graph import StateGraph, END

def route_after_search(state):
    if state["search_results"]: return "summarize"
    if state["retry_count"] >= 3: return "error"
    return "refine"

g = StateGraph(ResearchState)
for name, fn in [("query_planner", query_planner_node), ("search", search_node),
                 ("refine", refine_query_node), ("summarize", summarize_node),
                 ("answer", answer_node), ("error", error_node)]:
    g.add_node(name, fn)

g.set_entry_point("query_planner")
g.add_edge("query_planner", "search")
g.add_edge("refine", "search")       # cycle
g.add_edge("summarize", "answer")
g.add_edge("answer", END)
g.add_edge("error", END)
g.add_conditional_edges("search", route_after_search,
                        {"summarize": "summarize", "refine": "refine", "error": "error"})

research_graph = g.compile()
print("Compiled. Nodes:", list(research_graph.nodes))

## Run the Graph

### Test 1: Successful Research Query
A normal query that finds results on the first try. Watch the graph flow: planner, then search, then summarize, then answer, then END.

In [ ]:
def run_research(q):
    print(f"\n=== {q} ===")
    state = research_graph.invoke(fresh_state(q))
    print(f"Answer: {state['final_answer']}")
    return state

state = run_research("Latest advances in CAR-T cell therapy for leukemia?")
print(f"Retries: {state['retry_count']} | Refined: {state['refined_query']}")

<details>
<summary>Expected output (click to expand)</summary>

```
=== Latest advances in CAR-T cell therapy for leukemia? ===
  [planner] 'Latest advances in CAR-T cell therapy for leuk'
  [search] 'CAR-T cell therapy leukemia recent advances'
  [summarize] 2 results
  [answer]
Answer: Recent advances in CAR-T cell therapy for leukemia include...
Retries: 1 | Refined: CAR-T cell therapy leukemia recent advances
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Test 2: Query That Triggers Retry
The 'obscure' keyword causes the first search to fail, triggering the retry cycle: search (fail), then refine, then search (succeed), then summarize, then answer.

In [ ]:
# Trigger retry cycle
state2 = run_research("obscure specific topic no results expected")
print(f"Retries: {state2['retry_count']} | Answer: {state2['final_answer']}")

<details>
<summary>Expected output (click to expand)</summary>

```
=== obscure specific topic no results expected ===
  [planner] 'obscure specific topic no results expected'
  [search] 'obscure specific topic'
  [search] No results.
  [refine] retry=1
  [search] 'specific research topic scientific studies'
  [summarize] 2 results
  [answer]
Answer: ...
Retries: 2
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **Design a graph for your research domain.** What nodes would you need? Where would you add conditional edges and retry loops?
2. **The `REQUIRES_REVIEW` state routes to human oversight.** In your domain, what agent decisions should require human approval before proceeding?
3. **What's the maximum number of retries** that makes sense before giving up? How would you make this adaptive (e.g., fewer retries for expensive operations)?

## Summary

| LangGraph Concept | In This Graph |
|------------------|--------------|
| Node | query_planner, search, summarize, answer |
| Edge | search → summarize |
| Conditional edge | route_after_search() |
| Cycle (back-edge) | refine → search |
| END | answer → END, error → END |

**Next**: Lab 5 — evaluate output quality with LLM-as-judge.

---
*Agentic AI Science Playbook — Foundation Layer, Lab 4.*